# SAMATVA: free GPU render on Google Colab

Renders every photoreal plate of the SAMATVA soul film on Colab's free T4 graphics card. Frames are saved **straight into your Google Drive**, so nothing is lost if Colab disconnects.

**Before you start:**
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** When asked, allow access to Google Drive.
3. Keep this browser tab open. Free Colab disconnects after a while if the tab sits idle.

**If it disconnects:** reconnect and press **Run all** again. Frames already in Drive are skipped, and it carries on.

**When it finishes:** the frames are in **My Drive → SAMATVA_plates**. Share that folder, or the zip made at the end, as a link, and the film is assembled from it.

Expected time on one T4: about 1–2 hours for the hero settings, or about 15 minutes for the faster `film` settings.

## 1 · Settings

In [ ]:
QUALITY = 'hero'            # 'hero' = full HD, 64 samples, native 24 fps · 'film' = 720p, 10 samples, 8 fps (fast)
SHOTS   = ['reform', 'build', 'gen', 'loop', 'finale']
THERMAL = ['reform', 'build', 'gen']   # chapters that open in heat vision
DEVICE  = 'OPTIX'           # 'OPTIX' (fastest on T4) or 'CUDA' if OptiX fails
DRIVE_FOLDER = 'SAMATVA_plates'
REPO    = 'https://raw.githubusercontent.com/2025514153aashirwad-glitch/2025514153aashirwad-glitch.github.io/ccr-22a9ddef-0uao11'
HDRI_URL = 'https://dl.polyhaven.org/file/ph-assets/HDRIs/hdr/2k/brown_photostudio_02_2k.hdr'   # CC0, Poly Haven
BLENDER_URL = 'https://download.blender.org/release/Blender4.5/blender-4.5.16-linux-x64.tar.xz'

Q = {'hero': dict(res='1920x1080', fps=24, metal=64, thermal=16),
     'film': dict(res='1280x720',  fps=8,  metal=10, thermal=10)}[QUALITY]
LEN = {'reform': 14, 'build': 14, 'gen': 13, 'loop': 12, 'finale': 12}
TH_SECONDS = 5.5
print(QUALITY, Q)

## 2 · Connect Google Drive and check the graphics card

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
OUT = pathlib.Path('/content/drive/MyDrive') / DRIVE_FOLDER; OUT.mkdir(parents=True, exist_ok=True)
print('frames will be saved to', OUT)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: Runtime > Change runtime type > T4 GPU'

## 3 · Install Blender 4.5 LTS (about 2 minutes)

In [ ]:
import os
if not os.path.exists('/content/blender/blender'):
    !wget -q {BLENDER_URL} -O /content/blender.tar.xz
    !mkdir -p /content/blender && tar xf /content/blender.tar.xz -C /content/blender --strip-components=1 && rm /content/blender.tar.xz
!apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 libxfixes3 libgl1 > /dev/null 2>&1 || true
!/content/blender/blender -b --version | head -1

## 4 · Fetch the scene: traced Samatwam marks, loop path, shot script, studio light

In [ ]:
import urllib.request
A = pathlib.Path('/content/assets'); (A / 'vector').mkdir(parents=True, exist_ok=True)
files = [f'brand/logo/vector/{v}-{p}.svg' for v in ('sun', 'wheel', 'hexagon', 'octagon') for p in ('crescent', 'infinity', 'emblem', 'triangles')]
files += ['brand/logo/vector/sun-looppath.json']
for f in files:
    urllib.request.urlretrieve(f'{REPO}/{f}', A / 'vector' / pathlib.Path(f).name)
urllib.request.urlretrieve(f'{REPO}/films/samatva-soul/source/blender/soul_shots.py', A / 'soul_shots.py')
urllib.request.urlretrieve(HDRI_URL, A / 'studio.hdr')
print(len(list((A / 'vector').iterdir())), 'vector files ·', (A / 'studio.hdr').stat().st_size // 1_000_000, 'MB light map')

## 5 · Render, shot by shot, straight into Drive
Re-running this cell resumes: frames already in Drive are skipped. The progress line updates after every shot.

In [ ]:
import subprocess, time, math
env = dict(os.environ, SAMATVA_VEC=str(A / 'vector'), SAMATVA_HDRI=str(A / 'studio.hdr'), SAMATVA_DEVICE=DEVICE, SAMATVA_FORMAT='JPEG')
jobs = [(s, 'thermal', math.ceil(TH_SECONDS * Q['fps']), Q['thermal']) for s in THERMAL]
jobs += [(s, 'metal', LEN[s] * Q['fps'], Q['metal']) for s in SHOTS]
t_all = time.time()
for s, p, n, spp in jobs:
    out = OUT / f'{s}-{p}'; out.mkdir(exist_ok=True); (out / 'fps.txt').write_text(str(Q['fps']))
    done = len(list(out.glob('*.jpg')))
    if done >= n: print(f'{s:7s} {p:7s} already complete ({n} frames)'); continue
    t0 = time.time()
    r = subprocess.run(['/content/blender/blender', '-b', '--factory-startup', '-P', str(A / 'soul_shots.py'), '--', s, p, str(out),
                        '--frames', f'1-{n}', '--samples', str(spp), '--res', Q['res'], '--fps', str(Q['fps'])],
                       env=env, capture_output=True, text=True)
    ok = 'DONE' in r.stdout; got = len(list(out.glob('*.jpg')))
    print(f'{s:7s} {p:7s} {got:4d}/{n} frames  {"ok" if ok else "FAILED"}  {(time.time() - t0) / 60:5.1f} min', flush=True)
    if not ok: print(r.stdout[-2000:], r.stderr[-2000:]); break
print(f'total {(time.time() - t_all) / 60:.1f} min')

## 6 · Check: frame counts and a look at each shot

In [ ]:
from PIL import Image
from IPython.display import display
for d in sorted(OUT.iterdir()):
    if not d.is_dir(): continue
    fr = sorted(d.glob('*.jpg')); print(f'{d.name:16s} {len(fr):5d} frames')
    if fr: im = Image.open(fr[len(fr) // 2]); im.thumbnail((480, 270)); display(im)

## 7 · Optional: one zip in Drive, easy to share

In [ ]:
import shutil
z = shutil.make_archive(str(OUT.parent / 'samatva_plates'), 'zip', OUT)
print(z, round(os.path.getsize(z) / 1e9, 2), 'GB · right-click it in Drive → Share → Copy link')